# Milestone 3 — Tiền Xử Lý, Gán Nhãn & Trích Xuất Đặc Trưng (Colab Pipeline)

Notebook này phục vụ toàn bộ quy trình của **Người 1** trong Milestone 3:
1. **Mount Google Drive** & kết nối dữ liệu từ M1 (`references/range_*`) và M2 (`M2_processed/range_1/M2_range_1_tables.zip`).
2. **Giải nén & gom dữ liệu** thành các bảng chuẩn `data/tables/bibtex.jsonl` và `data/tables/references.jsonl`.
3. **Hỗ trợ gán nhãn thủ công (Mục 2)**: Giao diện bảng tương tác để nhóm kiểm tra 5 bài báo (≥ 20 refs/bài, tổng ≥ 100 entries).
4. **Gán nhãn tự động & Trích xuất 14 đặc trưng (Mục 3, 4, 5, 6)**: Tự động chạy thuật toán rule-based, trích xuất 14 features cho các cặp ứng viên, chia tập train / valid / test.
5. **Vẽ biểu đồ phân tích dữ liệu cho Poster A2**: Trực quan hóa phân bố các đặc trưng giữa cặp nhãn 1 và nhãn 0.
6. **Lưu các file Parquet bàn giao** (`train_pairs.parquet`, `valid_pairs.parquet`, `test_pairs.parquet`) trực tiếp về lại Google Drive.

## Bước 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive đã được kết nối thành công!")

## Bước 2: Thiết lập môi trường & Clone / Pull Code mới nhất

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Cấu hình thông tin repo của nhóm
REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # Thay đổi nếu dùng repo fork
BRANCH = "M1-Update-pipeline"                                 # Tên branch đang làm việc
PROJECT_DIR = Path("/content/arXir_project")

if not (PROJECT_DIR / ".git").exists():
    print(f"Đang clone nhánh {BRANCH} từ GitHub...")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print("Đang cập nhật code mới nhất từ GitHub...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Cài đặt các thư viện cần thiết
!pip install -q rapidfuzz pandas pyarrow scikit-learn matplotlib seaborn pytest

# Chạy kiểm thử xác nhận module Người 1 đã sẵn sàng
!python -m pytest tests/test_m3_matching.py -v
print("Hoàn tất thiết lập môi trường!")

## Bước 3: Nạp dữ liệu từ Google Drive (M2 Processed & M1 References)

Tự động giải nén `M2_range_1_tables.zip` và gom các file `references` từ các thư mục `range_1`..`range_5`.

In [ ]:
import shutil
import zipfile
import json
from pathlib import Path

# Khởi tạo thư mục tables nội bộ
LOCAL_TABLES = PROJECT_DIR / "data" / "tables"
LOCAL_TABLES.mkdir(parents=True, exist_ok=True)

# --- 1. Tìm và giải nén file M2_range_1_tables.zip ---
# Tìm kiếm trong MyDrive hoặc Shared with me
possible_m2_zips = list(Path("/content/drive").rglob("*M2_range*tables.zip"))
if not possible_m2_zips:
    # Thử đường dẫn mặc định trong Drive
    m2_zip_path = Path("/content/drive/MyDrive/M2_processed/range_1/M2_range_1_tables.zip")
else:
    m2_zip_path = possible_m2_zips[0]

print(f"File M2 tables zip: {m2_zip_path}")
if m2_zip_path.exists():
    print(f"Đang giải nén {m2_zip_path.name}...")
    with zipfile.ZipFile(m2_zip_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            if member.endswith("bibtex.jsonl"):
                with zip_ref.open(member) as source, open(LOCAL_TABLES / "bibtex.jsonl", "wb") as target:
                    shutil.copyfileobj(source, target)
                print("-> Đã trích xuất thành công: data/tables/bibtex.jsonl")
else:
    print(f"[Cảnh báo] Chưa tìm thấy {m2_zip_path}. Bạn hãy kiểm tra lại đường dẫn trong Drive.")

# --- 2. Gom dữ liệu Crawled References từ các range_1..range_5 ---
ref_files = list(Path("/content/drive/MyDrive/references").rglob("*.jsonl")) + \
            list(Path("/content/drive/MyDrive/arxiv-project-data").rglob("*references*.jsonl"))

target_ref = LOCAL_TABLES / "references.jsonl"
if ref_files:
    print(f"Tìm thấy {len(ref_files)} files references từ các range. Đang gộp dữ liệu...")
    seen_lines = set()
    with open(target_ref, "w", encoding="utf-8") as out_f:
        for rf in ref_files:
            with open(rf, "r", encoding="utf-8") as in_f:
                for line in in_f:
                    s_line = line.strip()
                    if s_line and s_line not in seen_lines:
                        seen_lines.add(s_line)
                        out_f.write(s_line + "\n")
    print(f"-> Đã tạo bảng tổng hợp: {target_ref} ({len(seen_lines)} dòng crawled references)")
else:
    print("[Cảnh báo] Chưa tìm thấy file references.jsonl trong Drive.")

## Bước 4: Hỗ trợ Gán nhãn thủ công (Mục 2)

Tìm các bài báo có $\ge 20$ BibTeX entries và hiển thị giao diện xem trực quan kèm điểm gợi ý tự động để các bạn trong nhóm xác nhận nhanh chóng.

In [ ]:
import pandas as pd
from src.m3_matching.manual_tool import find_candidate_manual_papers, export_manual_template

# 1. Tìm các bài báo đủ điều kiện (≥ 20 entries)
candidates = find_candidate_manual_papers()
print(f"Tìm thấy {len(candidates)} bài báo có ≥ 20 BibTeX entries và có references:")
for pid, b_cnt, r_cnt in candidates[:10]:
    print(f"  - Bài {pid}: {b_cnt} BibTeX entries | {r_cnt} References")

# 2. Xuất file mẫu cho 5 bài báo đầu tiên để nhóm làm thủ công
template_dir = PROJECT_DIR / "data" / "labels" / "manual_templates"
selected_5_papers = [pid for pid, _, _ in candidates[:5]]

for pid in selected_5_papers:
    out_csv = template_dir / f"{pid}.csv"
    export_manual_template(pid, out_csv)

print(f"\nĐã xuất 5 file CSV mẫu tại: {template_dir}")

# 3. Xem trước một bảng để kiểm tra trực quan ngay trên Colab
if selected_5_papers:
    demo_pid = selected_5_papers[0]
    df_preview = pd.read_csv(template_dir / f"{demo_pid}.csv")
    display(df_preview[["citation_key", "bib_title", "suggested_ref_title", "suggested_score", "groundtruth_arxiv_id"]].head(10))

## Bước 5: Chạy Trọn Gói Pipeline của Người 1

Thực hiện toàn bộ các bước:
- Làm sạch dữ liệu (`cleaning.py`)
- Gán nhãn tự động cho $\ge 10\%$ bài báo ngoài nhóm manual (`labelling.py`)
- Kiểm tra quota của thầy (`validate_label_quota`)
- Chia tập `train`, `valid`, `test` theo bài báo (`split.py`)
- Sinh bảng các cặp ứng viên kèm 14 đặc trưng và nhãn nhị phân 0/1 (`dataset.py`)

In [ ]:
from src.m3_matching.prepare_dataset import run_preparation_pipeline

# Cấu hình: k=2 nếu nhóm 2 người, seed=42 để kết quả chia tập hoàn toàn tái lập
K_MEMBERS = 2
RANDOM_SEED = 42

# Nếu chưa có file crawl thật, có thể thêm demo=True để chạy thử nghiệm
summary = run_preparation_pipeline(
    tables_dir=LOCAL_TABLES,
    labels_dir=PROJECT_DIR / "data" / "labels",
    k=K_MEMBERS,
    seed=RANDOM_SEED,
    demo=not (LOCAL_TABLES / "bibtex.jsonl").exists(),
)

print("\n=================== KẾT QUẢ XUẤT BẢN DỮ LIỆU ===================")
print(f"- Số lượng đặc trưng: {len(summary['feature_names'])} features")
print(f"- Tập Train: {summary['splits']['train']['n_pairs']:,} cặp ứng viên ({summary['splits']['train']['n_positives']} positive, {summary['splits']['train']['n_negatives']} negative)")
print(f"- Tập Valid: {summary['splits']['valid']['n_pairs']:,} cặp ứng viên ({summary['splits']['valid']['n_positives']} positive, {summary['splits']['valid']['n_negatives']} negative)")
print(f"- Tập Test : {summary['splits']['test']['n_pairs']:,} cặp ứng viên ({summary['splits']['test']['n_positives']} positive, {summary['splits']['test']['n_negatives']} negative)")
print("=================================================================")

## Bước 6: Trực Quan Hóa 14 Đặc Trưng & Phân Tích (Cho Poster A2 & Báo Cáo)

Vẽ biểu đồ phân bố độ tương đồng tiêu đề, tác giả, năm và lưu file ảnh để chèn vào Poster A2.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc tập dữ liệu cặp đã trích xuất đặc trưng
train_df = pd.read_parquet(LOCAL_TABLES / "train_pairs.parquet")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Phân bố độ tương đồng tiêu đề (title_token_sort)
sns.histplot(
    data=train_df, x="title_token_sort", hue="label",
    bins=30, kde=True, ax=axes[0], palette={0: "#e74c3c", 1: "#2ecc71"},
    common_norm=False, stat="density"
)
axes[0].set_title("Phân bố độ tương đồng Tiêu đề (Token Sort Ratio)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Score (0.0 .. 1.0)")
axes[0].legend(title="Nhãn", labels=["Trùng khớp (1)", "Khác bài (0)"])

# 2. Phân bố trùng khớp họ tác giả (author_jaccard)
sns.histplot(
    data=train_df, x="author_jaccard", hue="label",
    bins=20, kde=False, ax=axes[1], palette={0: "#e74c3c", 1: "#2ecc71"},
    common_norm=False, stat="density"
)
axes[1].set_title("Phân bố trùng khớp Họ Tác giả (Jaccard Overlap)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Jaccard Score")
axes[1].legend(title="Nhãn", labels=["Trùng khớp (1)", "Khác bài (0)"])

# 3. Phân bố chênh lệch năm công bố (year_diff <= 10)
year_df = train_df[train_df["year_diff"] < 20]
sns.boxplot(
    data=year_df, x="label", y="year_diff", ax=axes[2],
    palette={0: "#e74c3c", 1: "#2ecc71"}
)
axes[2].set_title("Độ lệch Năm Công bố theo Nhãn", fontsize=13, fontweight="bold")
axes[2].set_xticklabels(["Khác bài (0)", "Trùng khớp (1)"])
axes[2].set_ylabel("Chênh lệch năm tuyệt đối")

plt.tight_layout()
poster_fig_path = PROJECT_DIR / "poster_feature_distribution.png"
plt.savefig(poster_fig_path, dpi=300)
plt.show()
print(f"Đã lưu biểu đồ phân tích cho Poster tại: {poster_fig_path}")

## Bước 7: Lưu Kết Quả Parquet Ngược Về Google Drive

Tự động copy các file Parquet đã sinh vào thư mục Drive của nhóm để phục vụ bước huấn luyện mô hình.

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/M3_processed")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_files = [
    "train_pairs.parquet",
    "valid_pairs.parquet",
    "test_pairs.parquet",
    "dataset_pairs.parquet",
    "poster_feature_distribution.png"
]

print(f"Đang copy kết quả vào Google Drive: {DRIVE_OUTPUT_DIR}...")
for fname in export_files:
    src_p = LOCAL_TABLES / fname if (LOCAL_TABLES / fname).exists() else (PROJECT_DIR / fname)
    if src_p.exists():
        dst_p = DRIVE_OUTPUT_DIR / fname
        shutil.copy2(src_p, dst_p)
        print(f"  [✓] Đã lưu: {fname}")

print("\nHOÀN TẤT TOÀN BỘ QUY TRÌNH TIỀN XỬ LÝ & BÀN GIAO DỮ LIỆU MILESTONE 3!")